# Grafo bipartito de cargos y habilidades (vocabulario endógeno)

El grafo conecta cargos observados con los términos que el propio corpus usa para describir las vacantes. El vocabulario no proviene de una taxonomía externa: se construye con n-gramas de las descripciones, puntuados por su señal ocupacional (`frecuencia × log(K / ocupaciones)`), y después se recuperan vacante por vacante con un autómata Aho-Corasick que respeta los límites de palabra. Es una señal exploratoria de proximidad entre perfiles, no una recomendación laboral individual.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd()
if not (RAIZ / 'src').exists(): RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown
from src.grafos.metricas_grafo import crear_grafo_bipartito, resumen_red, ranking_grado, centralidad_aproximada, similitudes_ocupaciones, comunidades_ocupaciones, habilidades_puente, detalle_transicion, habilidades_caracteristicas
DIR = RAIZ / 'data/procesados/grafos'
faltantes = [p.name for p in [DIR/'nodos.csv', DIR/'aristas.csv', DIR/'manifiesto.json'] if not p.exists()]
if faltantes: raise FileNotFoundError('Primero ejecuta `python -m src.grafos.construir_grafo --repo pxtron/vacantes-colombia`. Faltan: ' + ', '.join(faltantes))
nodos = pd.read_csv(DIR/'nodos.csv')
aristas = pd.read_csv(DIR/'aristas.csv')
manifiesto = json.loads((DIR/'manifiesto.json').read_text(encoding='utf-8'))
display(Markdown(f'**Datos:** `{manifiesto["repo_id"]}` @ `{manifiesto["revision_resuelta"]}` · **vocabulario endógeno**: {manifiesto["terminos_vocabulario"]:,} términos sobre una muestra de {manifiesto["muestra_vocabulario"]:,} vacantes · **{manifiesto["vacantes_cualificadas"]:,}** vacantes cualificadas → **{manifiesto["nodos"]:,}** nodos y **{manifiesto["aristas"]:,}** aristas'))

**Datos:** `pxtron/vacantes-colombia` @ `1f820ba14350e28a3b74676ee3605ee47616ddb8` · **vocabulario endógeno**: 6,000 términos sobre una muestra de 60,000 vacantes · **90,015** vacantes cualificadas → **12,740** nodos y **367,075** aristas

## 1. Estructura de la red

In [2]:
G = crear_grafo_bipartito(nodos, aristas)
display(pd.Series(resumen_red(G), name='valor').to_frame())
display(Markdown('La densidad se calcula como aristas observadas sobre todas las combinaciones cargo–habilidad posibles. Una red dispersa es esperable cuando las habilidades son específicas.'))

,valor
nodos,12740.000000
ocupaciones,6742.000000
habilidades,5998.000000
aristas,367075.000000
densidad_bipartita,0.009077
componentes,1.000000


La densidad se calcula como aristas observadas sobre todas las combinaciones cargo–habilidad posibles. Una red dispersa es esperable cuando las habilidades son específicas.

In [3]:
top_skills = ranking_grado(G, 'habilidad', 20)
top_occ = ranking_grado(G, 'ocupacion', 20)
px.bar(top_skills.sort_values('grado'), x='grado', y='nombre', orientation='h', title='Habilidades conectadas con más cargos').show()
px.bar(top_occ.sort_values('grado'), x='grado', y='nombre', orientation='h', title='Cargos conectados con más habilidades').show()
display(Markdown('El grado mide diversidad de conexiones; la fuerza, disponible en las tablas, suma publicaciones y distingue amplitud de frecuencia.'))
display(top_skills); display(top_occ)
display(centralidad_aproximada(G, muestras=200, semilla=42).head(20))
display(Markdown('La betweenness aproximada identifica nodos que conectan zonas de la red; usa muestreo reproducible para evitar un cálculo cuadrático sobre el grafo completo.'))

El grado mide diversidad de conexiones; la fuerza, disponible en las tablas, suma publicaciones y distingue amplitud de frecuencia.

,node_id,nombre,grado,fuerza
3269,term_8acc5cd3743f371e,dias,844,7621.0
775,term_2082f857970a9592,coordinar,576,4266.0
5906,term_fcddc2aef8f12f0f,licencia,550,5262.0
2520,term_6a1906760aec88e4,compensatorio,544,4696.0
4536,term_c1e0addcbac9af82,recargos,499,4093.0
2447,term_6647a075828058b9,ingenieria,497,3495.0
780,term_20d6d66e45aae63a,traves,494,4754.0
5017,term_d608a2edbe35d268,minimo anos,487,3158.0
2903,term_79e3f3dc7b29c4b8,tienes experiencia,482,4117.0
4158,term_b226c0d40f34ba2e,prestaciones ley horario,478,3848.0


,node_id,nombre,grado,fuerza
1,occ_58867be587a81a80,Asesor Comercial,2449,35489.0
0,occ_ebfed71b51e97a6b,Auxiliar contable,2212,36067.0
2,occ_adc34fb5759add69,Auxiliar de bodega,2073,28644.0
3,occ_c8be11dcac8279b1,Auxiliar de cocina,1766,24503.0
4,occ_a96a0be8164142e2,Auxiliar de servicios generales,1556,18405.0
8,occ_b068e6a6e1784cba,Asesor/a comercial,1383,14753.0
5,occ_c55839466c8d755e,Analista contable,1257,12598.0
9,occ_17cf11fd8e260bdc,Auxiliar de mantenimiento,1257,11813.0
6,occ_433f12ffb31e9acf,Conductor,1124,13073.0
10,occ_7810e911d1afeca4,Auxiliar de farmacia,1040,11523.0


,node_id,nombre,tipo,betweenness
1,occ_58867be587a81a80,Asesor Comercial,ocupacion,0.081394
0,occ_ebfed71b51e97a6b,Auxiliar contable,ocupacion,0.049619
2,occ_adc34fb5759add69,Auxiliar de bodega,ocupacion,0.047886
3,occ_c8be11dcac8279b1,Auxiliar de cocina,ocupacion,0.035896
8,occ_b068e6a6e1784cba,Asesor/a comercial,ocupacion,0.024813
4,occ_a96a0be8164142e2,Auxiliar de servicios generales,ocupacion,0.024707
9,occ_17cf11fd8e260bdc,Auxiliar de mantenimiento,ocupacion,0.021825
5,occ_c55839466c8d755e,Analista contable,ocupacion,0.017986
16,occ_2507e7917a6ba63e,Ejecutivo Comercial,ocupacion,0.016893
10,occ_7810e911d1afeca4,Auxiliar de farmacia,ocupacion,0.013330


La betweenness aproximada identifica nodos que conectan zonas de la red; usa muestreo reproducible para evitar un cálculo cuadrático sobre el grafo completo.

## 2. Similitud, comunidades y habilidades puente

In [4]:
similitudes = similitudes_ocupaciones(nodos, aristas, vecinos=10, similitud_minima=.05)
comunidades = comunidades_ocupaciones(similitudes, semilla=42)
puentes = habilidades_puente(aristas, comunidades, nodos)
nombres = nodos.set_index('node_id')['nombre']
pares = similitudes.head(25).copy()
pares['ocupacion_a'] = pares['source'].map(nombres)
pares['ocupacion_b'] = pares['target'].map(nombres)
display(pares[['ocupacion_a','ocupacion_b','similitud_coseno']].style.format({'similitud_coseno':'{:.3f}'}))
display(puentes.head(20))
display(Markdown('El coseno usa prevalencias TF-IDF: premia habilidades compartidas y reduce el peso de las ubicuas. Las habilidades puente aparecen en varias comunidades, pero no implican por sí solas transferibilidad profesional.'))

,ocupacion_a,ocupacion_b,similitud_coseno
5,Auxiliar de Enfermería /Línea Agendar citas/Experiencia call center o asistencial - Proceso virtual inmediato!,Auxiliar de Enfermería Call Center – Contrato Indefinido + Bonos + Estabilidad Laboral - Experiencia en Líneas de Salud,1.000
6,Auxiliar de Enfermería /Línea Agendar citas/Experiencia call center o asistencial - Proceso virtual inmediato!,Tecnico Auxiliar de Enfermería Línea Salud – Contrato Indefinido + Beneficios Corporativos - !Únete a nuestro equipo!,1.000
50618,Asesor de Retención Bancaria / Lunes a Viernes / Comisiones - 6 Meses de experiencia,Asesor de Fidelización Bancaria / Ingresos + Prestaciones de Ley - 6 Meses de experiencia,1.000
0,Auxiliar de Enfermería /Línea Agendar citas/Experiencia call center o asistencial - Proceso virtual inmediato!,Técnico Auxiliar de Enfermería – Experiencia en Líneas de Salud - proceso virtual,1.000
8,Auxiliar de Enfermería /Línea Agendar citas/Experiencia call center o asistencial - Proceso virtual inmediato!,Auxiliar de Enfermería Línea Salud – Contrato Indefinido + Beneficios Corporativos - !Únete a nuestro equipo!,1.000
14904,"Multinacional Bancaria Busca Tecnólogos Comerciales - Tecnicos, Tecnologos, Profesionales en areas Administrativas","Tecnólogos con Experiencia en Call Center / Comisiones Sin Techo / Sector Bancario - Tecnicos, Tecnologos, Profesionales en areas Administrativas",1.000
14902,"Multinacional Bancaria Busca Tecnólogos Comerciales - Tecnicos, Tecnologos, Profesionales en areas Administrativas","Asesor Comercial para Importante Entidad Financiera / Perfil Administrativo - Tecnicos, Tecnologos, Profesionales en areas Administrativas",1.000
50561,Concentrix te paga un Técnico Bilingüe en el / Bachilleres sin experiencia !Estudia gratis!,Estudia sin costo y recibe apoyo económico – Técnico bilingüe con Concentrix,1.000
14907,"Multinacional Bancaria Busca Tecnólogos Comerciales - Tecnicos, Tecnologos, Profesionales en areas Administrativas","Tecnólogos con Talento Comercial / Gana Comisiones Sin Techo - Tecnicos, Tecnologos, Profesionales en areas Administrativas",1.000
28759,Agente servicio al cliente CHAT - auxiliar administrativo/contrato indefinido/respuesta inmediata,Call Center Bogotá Servicio al Cliente – SMLV + Prestaciones - Bachiller/respuesta inmedita,1.000


,node_id,comunidades,entropia,nombre
3269,term_8acc5cd3743f371e,47,3.088851,dias
780,term_20d6d66e45aae63a,43,2.924272,traves
1254,term_3344abb0b5b5f65e,42,3.224831,42 horas
3942,term_a8ef4a44c81a2cc1,41,3.220000,apasiona
1056,term_2afe4862ac1fa5b6,41,3.136540,42 horas semanales
704,term_1d45c13f525badae,39,2.956521,posibilidad
3539,term_97982ab0ee3122de,37,3.113223,soluciones
2903,term_79e3f3dc7b29c4b8,37,3.111798,tienes experiencia
4158,term_b226c0d40f34ba2e,37,3.044158,prestaciones ley horario
4760,term_cb4a215ceb504192,37,3.041780,bono


El coseno usa prevalencias TF-IDF: premia habilidades compartidas y reduce el peso de las ubicuas. Las habilidades puente aparecen en varias comunidades, pero no implican por sí solas transferibilidad profesional.

## 3. Subred de una ocupación

In [5]:
# Los nombres son títulos normalizados sin homologar: incluyen ruido promocional
# ('!!Urgente!! ...'), por eso la ejemplo por defecto es la ocupación con más
# vacantes y no la primera alfabéticamente. Cambia OCUPACION_SELECCIONADA por
# cualquier otra de `opciones` y vuelve a ejecutar la celda.
opciones = nodos.loc[nodos.tipo.eq('ocupacion'), ['node_id','nombre','frecuencia']].sort_values('nombre')
OCUPACION_SELECCIONADA = opciones.nlargest(1, 'frecuencia')['nombre'].iloc[0]
display(opciones.nlargest(30, 'frecuencia'))
occ_id = opciones.loc[opciones.nombre.eq(OCUPACION_SELECCIONADA), 'node_id'].iloc[0]
skills_top = aristas[aristas.source.eq(occ_id)].nlargest(10, 'peso')['target'].tolist()
vecinos = pd.concat([similitudes[similitudes.source.eq(occ_id)].rename(columns={'target':'vecino'}), similitudes[similitudes.target.eq(occ_id)].rename(columns={'source':'vecino'})]).nlargest(5, 'similitud_coseno')['vecino'].tolist()
seleccion = {occ_id, *skills_top, *vecinos}
for vecino in vecinos: seleccion.update(aristas[aristas.source.eq(vecino)].nlargest(5, 'peso')['target'])
sub = G.subgraph(seleccion).copy()
pos = nx.spring_layout(sub, seed=42, weight='peso')
edge_x, edge_y = [], []
for a,b in sub.edges(): edge_x += [pos[a][0],pos[b][0],None]; edge_y += [pos[a][1],pos[b][1],None]
fig = go.Figure(go.Scatter(x=edge_x,y=edge_y,mode='lines',line={'color':'#bbb','width':1},hoverinfo='none'))
for tipo, color in [('ocupacion','#123c63'),('habilidad','#e8b93a')]:
    ns=[n for n,d in sub.nodes(data=True) if d['tipo']==tipo]
    fig.add_trace(go.Scatter(x=[pos[n][0] for n in ns],y=[pos[n][1] for n in ns],mode='markers+text',text=[sub.nodes[n]['nombre'] for n in ns],textposition='top center',name=tipo,marker={'size':16 if tipo=='ocupacion' else 10,'color':color}))
fig.update_layout(title=f'Subred: {OCUPACION_SELECCIONADA}',showlegend=True,height=700,xaxis={'visible':False},yaxis={'visible':False})
fig.show()
display(habilidades_caracteristicas(occ_id, nodos, aristas, n=10).style.format({'puntaje_tfidf':'{:.3f}'}))
display(Markdown('La subred limita habilidades y vecinos para conservar legibilidad; la tabla TF-IDF destaca habilidades frecuentes en este cargo pero menos ubicuas en los demás.'))

,node_id,nombre,frecuencia
0,occ_ebfed71b51e97a6b,Auxiliar contable,1270
1,occ_58867be587a81a80,Asesor Comercial,883
2,occ_adc34fb5759add69,Auxiliar de bodega,860
3,occ_c8be11dcac8279b1,Auxiliar de cocina,758
4,occ_a96a0be8164142e2,Auxiliar de servicios generales,543
5,occ_c55839466c8d755e,Analista contable,438
6,occ_433f12ffb31e9acf,Conductor,392
7,occ_cd5a3599e51cbf3e,Auxiliar administrativo,341
8,occ_b068e6a6e1784cba,Asesor/a comercial,339
9,occ_17cf11fd8e260bdc,Auxiliar de mantenimiento,326


,habilidad,puntaje_tfidf
0,archivo,0.787
1,soportes,0.751
2,finanzas,0.721
3,conciliacion,0.687
4,intermedio,0.625
5,estados,0.620
6,precision,0.604
7,manejo excel,0.602
8,apoyar procesos,0.589
9,utilizando,0.568


La subred limita habilidades y vecinos para conservar legibilidad; la tabla TF-IDF destaca habilidades frecuentes en este cargo pero menos ubicuas en los demás.

## 4. Señal exploratoria de transición

In [6]:
if vecinos:
    transicion = detalle_transicion(occ_id, vecinos[0], aristas, nodos)
    display(pd.Series(transicion).to_frame('resultado'))
else:
    display(Markdown('No hay una ocupación vecina que supere el umbral de similitud para el perfil seleccionado.'))

,resultado
origen,Auxiliar contable
destino,Analista contable
habilidades_compartidas,"[profesional contaduria, profesional contaduri..."
habilidades_adicionales_observadas,"[cargo analista, buscamos analista, encuentra ..."
aviso,Señal exploratoria de proximidad entre perfile...


## Limitaciones

El vocabulario es endógeno: los términos son n-gramas literales de las descripciones, así que pueden recoger hábitos de redacción o plantillas de un mismo empleador (el filtro de redundancia y la señal ocupacional lo mitigan, no lo eliminan). La coincidencia es literal y sin contexto: no unifica sinónimos ni variantes morfológicas ("vendedor" y "comercial" quedan como términos distintos) y no disambigua sentidos según el contexto. Los términos aún no se validan contra una taxonomía externa (ESCO es una fase pendiente), así que el vocabulario no es interoperable con otros estudios. Los cargos son títulos normalizados, no ocupaciones homologadas. Las comunidades y rutas describen coocurrencia en vacantes, no competencias reales de una persona.